# 01 · Exploratory analysis

**Project FORESIGHT — NorthBay Living**

The narrative behind deliverable D2. The figures and findings that reach the memo are
generated by `foresight.eda`; this notebook is where the questions were asked.

Run `python scripts/01_run_pipeline.py` first — everything here reads the cleaned outputs
rather than the raw extracts, so the cleaning decisions are already applied and visible in
`cleaning_report.parquet`.

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

from foresight.config import get_settings
from foresight.eda import PALETTE, apply_house_style

warnings.filterwarnings("ignore", category=FutureWarning)
apply_house_style()
pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 40)

settings = get_settings()
settings.processed_dir

In [ ]:
panel = pd.read_parquet(settings.processed_dir / "weekly_panel.parquet")
daily = pd.read_parquet(settings.processed_dir / "analysis_ready.parquet")
cleaning = pd.read_parquet(settings.processed_dir / "cleaning_report.parquet")

print(f"{panel['sku_id'].nunique()} SKUs x {panel['week_start'].nunique()} weeks "
      f"= {len(panel):,} rows")
print(f"{panel['week_start'].min().date()} to {panel['week_start'].max().date()}")

## 1. What did the cleaner actually have to fix?

The pipeline records every decision with its rationale, so this is the audit trail rather
than a summary written after the fact.

In [ ]:
cleaning.sort_values("rows_affected", ascending=False)[
    ["severity", "table", "issue", "rows_affected", "resolution"]
]

The single largest number here is the densification step. The client's export only contains
rows where a sale happened, so **77,503 zero-demand SKU-days were simply missing**. Any
lag or rolling feature built on the raw file would be silently shifted.

## 2. Seasonality — how large is the festive swing?

In [ ]:
weekly = panel.groupby("week_start", as_index=False)["units"].sum()

figure, axes = plt.subplots(figsize=(11, 3.4))
axes.plot(weekly["week_start"], weekly["units"], color=PALETTE["harbour_700"], linewidth=1.5)
axes.fill_between(weekly["week_start"], weekly["units"], color=PALETTE["harbour_500"], alpha=0.10)
axes.set_title("Total weekly demand")
axes.set_ylabel("Units")
plt.show()

peak, trough = weekly["units"].max(), weekly["units"].min()
print(f"peak {peak:,.0f}  trough {trough:,.0f}  ratio {peak / trough:.1f}x")

A swing this large is the central fact of the assortment. It rules out any planning rule
based on an annual average, and it is why the baseline the model must beat is
**seasonal-naive** rather than a moving average.

## 3. Do categories peak at the same time?

In [ ]:
frame = panel.copy()
frame["iso_week"] = frame["week_start"].dt.isocalendar()["week"].astype("int64")

profile = frame.groupby(["category", "iso_week"])["units"].mean().reset_index()
category_mean = frame.groupby("category")["units"].mean()
profile["index"] = profile.apply(
    lambda row: row["units"] / category_mean[row["category"]], axis=1
)

pivot = profile.pivot(index="iso_week", columns="category", values="index")
pivot.plot(figsize=(11, 3.8), linewidth=1.4, colormap=None)
plt.axhline(1.0, color=PALETTE["ink_3"], linestyle="--", linewidth=0.8)
plt.title("Seasonal index by category (1.0 = that category's average week)")
plt.ylabel("Index")
plt.legend(fontsize=8, ncols=3)
plt.show()

pivot.idxmax().rename("peak ISO week").to_frame()

**They do not.** Most categories peak around week 44 (Diwali), but Storage & Organisation
peaks in January instead. A single portfolio-level seasonal curve would misprice both.
This is what the ensemble's category-pooled seasonal profile component exists to capture.

## 4. Where is the revenue, and how long is the tail?

In [ ]:
by_sku = (
    panel.groupby("sku_id")["revenue"].sum().sort_values(ascending=False).reset_index()
)
by_sku["cumulative"] = by_sku["revenue"].cumsum() / by_sku["revenue"].sum()

for share in (0.5, 0.8, 0.9):
    count = int((by_sku["cumulative"] <= share).sum() + 1)
    print(f"{share:.0%} of revenue comes from {count} SKUs ({count / len(by_sku):.0%})")

## 5. How intermittent is the tail?

In [ ]:
zero_share = panel.groupby("sku_id")["units"].apply(lambda values: (values <= 0).mean())

print(f"median share of zero weeks: {zero_share.median():.1%}")
print(f"SKUs zero in >= 35% of weeks: {(zero_share >= 0.35).sum()} of {len(zero_share)}")

zero_share.plot(kind="hist", bins=30, figsize=(9, 3), color=PALETTE["harbour_500"])
plt.title("Share of weeks with no sale, per SKU")
plt.xlabel("Share of zero weeks")
plt.show()

This is the finding that drives two later decisions:

1. **WAPE, not MAPE.** MAPE is undefined on a zero week and explodes either side of one.
2. **The tail needs its own estimator.** A single model trained on absolute error will
   predict near-zero for these and be accurate on average while being useless to plan with.

## 6. Do promotions actually move demand?

In [ ]:
frame = panel.copy()
frame["on_promo"] = frame["promo_days"] > 0

effect = (
    frame.groupby(["category", "on_promo"])["units"].mean()
    .unstack()
    .rename(columns={False: "normal week", True: "promo week"})
)
effect["uplift"] = effect["promo week"] / effect["normal week"] - 1.0
effect.sort_values("uplift", ascending=False).style.format(
    {"normal week": "{:.1f}", "promo week": "{:.1f}", "uplift": "{:.0%}"}
)

Yes, and by materially different amounts per category — so a single blanket uplift
assumption would misprice the stock build for every category at once. Promotion dates come
from the published calendar, so they are **known in advance** and are a legitimate forecast
input rather than something to react to.

---

Next: [`02_baseline.ipynb`](02_baseline.ipynb) — fixing the metric and building the bar.